In [1]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
from joblib import Parallel, delayed

# CONFIGURAÇÕES

In [ ]:
QUANTIZATION_LEVELS = [256, 128, 64, 32, 16]
DISTANCES = [1, 2, 4, 8]
ANGLES = [0, np.pi/4, np.pi/2, 3*np.pi/4]
GLCM_PROPS = ["contrast", "dissimilarity", "homogeneity", "energy", "correlation", "ASM"]

LBP_RADII = [1, 2, 3]
LBP_METHOD = "uniform"

DATASET_PATH = "../../dataset/Data/Data"
IMG_SIZE = 512  # r512
SAVE_DIR = "./saved"
ZONE_DIVISIONS = [(1,1),(2,2),(3,3),(4,4)]  # divisões espaciais
BATCH_SIZE = 500  # salvar por batch

# FUNÇÕES AUXILIARES

In [ ]:
def normalize_50_50(gray):
    med = np.median(gray)
    return np.where(gray < med, 0, 255).astype(np.uint8)

def quantize_gray(image, levels):
    return np.floor(image / (256 / levels)).astype(np.uint8)

def resize_with_padding(img, size=IMG_SIZE):
    """
    Redimensiona imagem preservando proporção e adiciona padding preto.
    Se imagem menor que size, só aplica padding.
    """
    h, w = img.shape[:2]

    if max(h, w) <= size:
        # imagem menor → só padding
        padded = np.zeros((size, size), dtype=np.uint8)
        pad_top = (size - h) // 2
        pad_left = (size - w) // 2
        padded[pad_top:pad_top+h, pad_left:pad_left+w] = img
        return padded
    else:
        # imagem maior → reduzir proporcionalmente
        scale = size / max(h, w)
        new_w, new_h = int(w * scale), int(h * scale)
        resized = cv2.resize(img, (new_w, new_h))
        padded = np.zeros((size, size), dtype=np.uint8)
        pad_top = (size - new_h) // 2
        pad_left = (size - new_w) // 2
        padded[pad_top:pad_top+new_h, pad_left:pad_left+new_w] = resized
        return padded

def split_image(img, rows, cols):
    h, w = img.shape
    zones = []
    row_height = h // rows
    col_width = w // cols
    for i in range(rows):
        for j in range(cols):
            zone = img[i*row_height:(i+1)*row_height, j*col_width:(j+1)*col_width]
            zones.append(zone)
    return zones

# FUNÇÕES DE FEATURES

In [ ]:
def extract_glcm_features(gray):
    feats = []
    for levels in QUANTIZATION_LEVELS:
        q = quantize_gray(gray, levels)
        glcm = graycomatrix(q, distances=DISTANCES, angles=ANGLES, levels=levels, symmetric=True, normed=True)
        for prop in GLCM_PROPS:
            feats.extend(graycoprops(glcm, prop).flatten())
    return np.array(feats, dtype=np.float32)

def extract_glcm_zones(gray, rows, cols):
    feats = []
    for z in split_image(gray, rows, cols):
        feats.extend(extract_glcm_features(z))
    return np.array(feats, dtype=np.float32)

def extract_lbp_features(gray, radius):
    points = 8 * radius
    lbp = local_binary_pattern(gray, points, radius, LBP_METHOD)
    hist, _ = np.histogram(lbp, bins=np.arange(0, points+3), range=(0, points+2), density=True)
    return hist.astype(np.float32)

def extract_lbp_zones(gray, radius, rows, cols):
    feats = []
    for z in split_image(gray, rows, cols):
        feats.extend(extract_lbp_features(z, radius))
    return np.array(feats, dtype=np.float32)

# CARREGAR IMAGENS

In [ ]:
def load_images(root=DATASET_PATH, size=IMG_SIZE):
    X = []
    y = []
    classes = sorted(os.listdir(root))
    label_map = {cls:i for i,cls in enumerate(classes)}
    for label in classes:
        folder = os.path.join(root, label)
        if not os.path.isdir(folder): continue
        print(f"Carregando classe: {label}")
        for imgname in os.listdir(folder):
            path = os.path.join(folder, imgname)
            img = cv2.imread(path)
            if img is None: continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            img = resize_with_padding(img, size)
            X.append(img)
            y.append(label_map[label])
    return np.array(X), np.array(y)

# FUNÇÃO PARA UMA IMAGEM (PARALELIZÁVEL)

In [ ]:
def process_image(img):
    from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
    import numpy as np

    feats_glcm = {}
    feats_lbp = {}

    # GLCM
    for rows, cols in ZONE_DIVISIONS:
        feats_glcm[f"{rows}x{cols}"] = extract_glcm_zones(img, rows, cols)

    # LBP
    img_5050 = normalize_50_50(img)
    for r in LBP_RADII:
        for rows, cols in ZONE_DIVISIONS:
            feats_lbp[f"normal_r{r}_{rows}x{cols}"] = extract_lbp_zones(img, r, rows, cols)
            feats_lbp[f"5050_r{r}_{rows}x{cols}"] = extract_lbp_zones(img_5050, r, rows, cols)

    return feats_glcm, feats_lbp

# EXECUÇÃO PRINCIPAL

In [ ]:
if __name__=="__main__":
    os.makedirs(SAVE_DIR, exist_ok=True)

    # 1) Carregar imagens
    X_img, y = load_images(DATASET_PATH)
    print(f"\nTotal de imagens: {len(X_img)}")

    # 2) Processar em paralelo e salvar por batch
    for start in range(0, len(X_img), BATCH_SIZE):
        end = min(start + BATCH_SIZE, len(X_img))
        batch_imgs = X_img[start:end]
        print(f"\nProcessando batch {start} a {end}...")

        results = Parallel(n_jobs=-1)(delayed(process_image)(img) for img in tqdm(batch_imgs))

        # Separar GLCM e LBP por batch
        glcm_batch = {f"{r}x{c}":[] for r,c in ZONE_DIVISIONS}
        lbp_batch = {}
        for r in LBP_RADII:
            for norm in ["normal","5050"]:
                for rows, cols in ZONE_DIVISIONS:
                    key = f"{norm}_r{r}_{rows}x{cols}"
                    lbp_batch[key] = []

        for g,l in results:
            for k in g: glcm_batch[k].append(g[k])
            for k in l: lbp_batch[k].append(l[k])

        # Converter para np.array e salvar
        for k,v in glcm_batch.items():
            np.save(os.path.join(SAVE_DIR, f"X_glcm_{k}_batch{start}.npy"), np.array(v))
        for k,v in lbp_batch.items():
            np.save(os.path.join(SAVE_DIR, f"X_lbp_{k}_batch{start}.npy"), np.array(v))

    # 3) Salvar labels
    np.save(os.path.join(SAVE_DIR, "y.npy"), y)
    print("\n✔ Features salvas com batchs em 'saved/'")


Carregando classe: Ace
Carregando classe: Akainu
Carregando classe: Brook
Carregando classe: Chopper
Carregando classe: Crocodile
Carregando classe: Franky
Carregando classe: Jinbei
Carregando classe: Kurohige
Carregando classe: Law
Carregando classe: Luffy
Carregando classe: Mihawk
Carregando classe: Nami
Carregando classe: Rayleigh
Carregando classe: Robin
Carregando classe: Sanji
Carregando classe: Shanks
Carregando classe: Usopp
Carregando classe: Zoro

Total de imagens: 11737

Processando batch 0 a 500...


100%|██████████| 500/500 [50:24<00:00,  6.05s/it]



Processando batch 500 a 1000...


 15%|█▍        | 74/500 [07:26<43:31,  6.13s/it]